In [ ]:
"""
stage2_hawkes.py -- Stage 2 (+3): discrete-time logistic Hawkes (notebook version).

RULES
  S2.1  p_t = P(is_target[t] | info <= t-1). Event-count features use event_bar
        in lag bins with min lag 1; age/tod computed from info <= t-1 (S2.3).
  S2.2  Lag bins (lo, hi) inclusive from bin_edges: (1,2,3,5,...) ->
        [1,1],[2,2],[3,3],[4,5],...
  S2.3  age(t) = t - (last target bar strictly before t), session-local;
        no prior target -> age = local_t + 1. Never reads is_target[t].
  S2.4  All features are within-session => session-date splits need no embargo.
  S2.5  pcont(t, H) = prod_{h=1..H} (1 - p_{t+h}) with the event set frozen at
        <= t (counterfactual: no new oscillator or target events after t).
        Truncated at session end; h_used reported. pbreak = 1 - pcont.
  S2.6  Exogenous events with opposing == 0 dropped; TICK_HMA dropped.
  S2.7  Warm bars excluded from fit, predict and pcont query points.
  S2.8  Plain L2 logistic regression (lbfgs), no class weighting.
  S2.9  bootstrap=N: session-level resampling with replacement, percentile CIs
        on kernel coefficients.
  S2.10 tod bins read raw hour*60+minute against TOD_START_MIN / TOD_BIN_MIN /
        N_TOD. No tz handling: timestamps must already be session-local and the
        constants must match the session template -- verify at data prep.
"""

import json
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression

TOD_START_MIN = 540      # first session minute (9:00). Set to your template.
TOD_BIN_MIN = 30
N_TOD = 14               # bins covering the session. Set to your template.

STREAMS = ["MNQ_D1", "MNQ_D2", "TICK_D1", "TICK_D2"]


def _make_classes():
    cls = []
    for s in STREAMS:
        cls.append((s, "opp"))
        cls.append((s, "conf"))
    cls.append(("MNQ_JMA_SELF", "all"))
    return cls


def _bins_from_edges(edges):
    bins, lo = [], 1
    for e in edges:
        bins.append((lo, int(e)))
        lo = int(e) + 1
    return bins


def _age_bins_from_edges(edges):
    b = _bins_from_edges(edges)
    b.append((int(edges[-1]) + 1, 10 ** 9))
    return b


class Featurizer:
    def __init__(self, bars, events,
                 bin_edges=(1, 2, 3, 5, 8, 13, 21, 34),
                 age_edges=(1, 2, 3, 5, 8, 13, 21, 34, 55, 89)):
        self.bin_edges = list(bin_edges)
        self.age_edges = list(age_edges)
        self.bins = _bins_from_edges(bin_edges)
        self.age_bins = _age_bins_from_edges(age_edges)
        self.classes = _make_classes()
        self.feature_names = (
            [f"{s}|{c}|lag{lo}_{hi}" for (s, c) in self.classes for (lo, hi) in self.bins]
            + [f"age|{lo}_{hi}" for (lo, hi) in self.age_bins]
            + [f"tod|{k}" for k in range(N_TOD)]
        )
        self.n_feat = len(self.feature_names)

        ev = events[events["stream"].isin([c[0] for c in self.classes])]
        ev = ev[~((ev["stream"] != "MNQ_JMA_SELF") & (ev["opposing"] == 0))]   # S2.6
        evg = dict(tuple(ev.groupby("session_date")))

        self.sessions = []
        bars = bars.sort_values("bar_index").reset_index(drop=True)
        for sess, g in bars.groupby("session_date", sort=True):
            n = len(g)
            first = int(g["bar_index"].iloc[0])
            tgt = g["is_target"].to_numpy()
            warm = g["warm"].to_numpy()
            ts = pd.DatetimeIndex(g["timestamp"])
            mins = ts.hour.to_numpy() * 60 + ts.minute.to_numpy() - TOD_START_MIN   # S2.10
            tod = np.clip(mins // TOD_BIN_MIN, 0, N_TOD - 1).astype(np.int16)
            lt_incl = np.maximum.accumulate(np.where(tgt, np.arange(n), -1))
            P = {}
            e = evg.get(sess)
            for (s, c) in self.classes:
                if e is None:
                    loc = np.empty(0, np.int64)
                else:
                    if s == "MNQ_JMA_SELF":
                        sub = e[e["stream"] == s]
                    else:
                        want = 1 if c == "opp" else -1
                        sub = e[(e["stream"] == s) & (e["opposing"] == want)]
                    loc = sub["event_bar"].to_numpy() - first
                ind = np.zeros(n, np.int32)
                ind[loc] = 1
                P[(s, c)] = np.concatenate(([0], np.cumsum(ind)))              # P[i] = count pos < i
            self.sessions.append(dict(
                sess=sess, first=first, n=n, tgt=tgt, warm=warm, tod=tod,
                lt_incl=lt_incl, P=P,
                bar_index=g["bar_index"].to_numpy(),
                timestamp=g["timestamp"].to_numpy()))

    def _fill(self, S, t, out):
        n = S["n"]
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:                                         # S2.1, S2.2
                b = np.clip(t - lo + 1, 0, n)
                a = np.clip(t - hi, 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S2.3
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][t]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _fill_frozen(self, S, q, h, out):
        """S2.5: features at t = q+h using only events/targets with pos <= q."""
        n = S["n"]
        t = q + h
        cap = q + 1
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:
                b = np.clip(np.minimum(t - lo + 1, cap), 0, n)
                a = np.clip(np.minimum(t - hi, cap), 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = S["lt_incl"][q]
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][np.minimum(t, n - 1)]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _selected(self, date_from, date_to):
        for S in self.sessions:
            d = str(S["sess"])
            if date_from is not None and d < date_from:
                continue
            if date_to is not None and d > date_to:
                continue
            yield S

    def build(self, date_from=None, date_to=None):
        sel = []
        total = 0
        for S in self._selected(date_from, date_to):
            t = np.nonzero(~S["warm"])[0]                                      # S2.7
            sel.append((S, t))
            total += len(t)
        X = np.zeros((total, self.n_feat), np.float32)
        y = np.zeros(total, np.int8)
        bar_index = np.zeros(total, np.int64)
        sess_id = np.zeros(total, np.int32)
        timestamp = np.zeros(total, "datetime64[ns]")
        ofs = 0
        for sid, (S, t) in enumerate(sel):
            m = len(t)
            self._fill(S, t, X[ofs:ofs + m])
            y[ofs:ofs + m] = S["tgt"][t]
            bar_index[ofs:ofs + m] = S["bar_index"][t]
            sess_id[ofs:ofs + m] = sid
            timestamp[ofs:ofs + m] = S["timestamp"][t]
            ofs += m
        meta = pd.DataFrame({"bar_index": bar_index, "timestamp": timestamp,
                             "sess_id": sess_id, "is_target": y.astype(bool)})
        return X, y, meta


def fit_hawkes(fz, train_start=None, train_end=None, c=100.0, bootstrap=0,
               out_dir=None):
    X, y, meta = fz.build(train_start, train_end)
    model = LogisticRegression(C=c, solver="lbfgs", max_iter=2000)             # S2.8
    model.fit(X, y)
    p = model.predict_proba(X)[:, 1]
    eps = 1e-12
    ll = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
    r = y.mean()
    ll_const = -(r * np.log(r) + (1 - r) * np.log(1 - r))

    bundle = dict(model=model, feature_names=fz.feature_names,
                  bin_edges=fz.bin_edges, age_edges=fz.age_edges, c=c,
                  train_start=train_start, train_end=train_end)

    coef = model.coef_[0]
    kern = pd.DataFrame({"feature": fz.feature_names, "coef": coef})
    kern["group"] = kern["feature"].str.split("|").str[0]
    kern["cls"] = kern["feature"].str.split("|").str[1]

    if bootstrap > 0:                                                          # S2.9
        rng = np.random.default_rng(7)
        sids = np.unique(meta["sess_id"].to_numpy())
        sid_arr = meta["sess_id"].to_numpy()
        rows_by_sid = {s: np.nonzero(sid_arr == s)[0] for s in sids}
        coefs = np.zeros((bootstrap, fz.n_feat))
        for b in range(bootstrap):
            pick = rng.choice(sids, size=len(sids), replace=True)
            rows = np.concatenate([rows_by_sid[s] for s in pick])
            mb = LogisticRegression(C=c, solver="lbfgs", max_iter=2000)
            mb.fit(X[rows], y[rows])
            coefs[b] = mb.coef_[0]
        kern["coef_lo"] = np.percentile(coefs, 2.5, axis=0)
        kern["coef_hi"] = np.percentile(coefs, 97.5, axis=0)

    bundle["kernels"] = kern
    if out_dir is not None:
        joblib.dump(bundle, f"{out_dir}/hawkes_model.joblib")
        kern.to_csv(f"{out_dir}/kernels.csv", index=False)

    summary = dict(n_rows=int(len(y)), n_targets=int(y.sum()),
                   base_rate=float(r), logloss_insample=float(ll),
                   logloss_const=float(ll_const),
                   intercept=float(model.intercept_[0]), n_features=fz.n_feat)
    print(json.dumps(summary, indent=2))
    return bundle


def predict_hawkes(fz, bundle, start=None, end=None, out_file=None):
    X, y, meta = fz.build(start, end)
    meta = meta.drop(columns=["sess_id"]).copy()
    meta["p"] = bundle["model"].predict_proba(X)[:, 1]
    if out_file is not None:
        meta.to_parquet(out_file, index=False)
    return meta


def pcont_hawkes(fz, bundle, at_bar_index, horizon=8, out_file=None):
    model = bundle["model"]
    want = np.sort(np.asarray(at_bar_index, dtype=np.int64))

    out_bar, out_pc, out_hu = [], [], []
    for S in fz.sessions:
        lo, hi = S["bar_index"][0], S["bar_index"][-1]
        w = want[(want >= lo) & (want <= hi)]
        if len(w) == 0:
            continue
        q = w - S["first"]
        q = q[~S["warm"][q]]                                                   # S2.7
        if len(q) == 0:
            continue
        n = S["n"]
        logsurv = np.zeros(len(q))
        h_used = np.zeros(len(q), np.int32)
        buf = np.zeros((len(q), fz.n_feat), np.float32)
        for h in range(1, horizon + 1):
            valid = q + h <= n - 1
            if not valid.any():
                break
            iv = np.nonzero(valid)[0]
            sub = buf[:len(iv)]
            fz._fill_frozen(S, q[iv], h, sub)                                  # S2.5
            p = model.predict_proba(sub)[:, 1]
            logsurv[iv] += np.log1p(-p)
            h_used[iv] += 1
        out_bar.append(q + S["first"])
        out_pc.append(np.exp(logsurv))
        out_hu.append(h_used)

    res = pd.DataFrame({"bar_index": np.concatenate(out_bar),
                        "pcont": np.concatenate(out_pc),
                        "h_used": np.concatenate(out_hu)})
    res["pbreak"] = 1.0 - res["pcont"]
    res["H"] = horizon
    if out_file is not None:
        res.to_parquet(out_file, index=False)
    return res


def plot_kernels(fz, bundle, out_path=None):
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    coef = bundle["model"].coef_[0]
    ks = fz.classes
    nc = 3
    nr = (len(ks) + nc - 1) // nc
    fig = make_subplots(rows=nr, cols=nc, subplot_titles=[f"{s}|{c}" for (s, c) in ks])
    centers = [0.5 * (lo + hi) for (lo, hi) in fz.bins]
    nb = len(fz.bins)
    for i, (s, c) in enumerate(ks):
        r, cix = i // nc + 1, i % nc + 1
        w = coef[i * nb:(i + 1) * nb]
        fig.add_trace(go.Scatter(x=centers, y=w, mode="lines+markers",
                                 line_shape="hv", showlegend=False), row=r, col=cix)
        fig.add_trace(go.Scatter(x=[centers[0], centers[-1]], y=[0, 0], mode="lines",
                                 line=dict(color="black", dash="dot", width=1),
                                 showlegend=False), row=r, col=cix)
    fig.update_layout(height=320 * nr, width=1400, title="Fitted kernels (logit weight vs lag)")
    if out_path is not None:
        fig.write_html(out_path)
    fig.show()
    return fig